In [1]:
import numpy as np
from numba import cuda

# CUDA Kernel
@cuda.jit
def row_sum_kernel(matrix, row_sums):
    row = cuda.grid(1)

    if row < matrix.shape[0]:
        total = 0
        for col in range(matrix.shape[1]):
            total += matrix[row, col]

        row_sums[row] = total


# Matrix size
rows = 1024
cols = 1024

# Create input matrix
matrix = np.random.randint(1, 10, size=(rows, cols)).astype(np.int32)

# Output array
row_sums = np.zeros(rows, dtype=np.int32)

# Transfer data to GPU
d_matrix = cuda.to_device(matrix)
d_row_sums = cuda.to_device(row_sums)

# Configure kernel launch
threads_per_block = 256
blocks_per_grid = (rows + threads_per_block - 1) // threads_per_block

# Launch kernel
row_sum_kernel[blocks_per_grid, threads_per_block](d_matrix, d_row_sums)

# Copy result back to CPU
row_sums = d_row_sums.copy_to_host()

# Display first 10 row sums
print("First 10 Row Sums:")
print(row_sums[:10])

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:748: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


First 10 Row Sums:
[5037 5099 5011 5120 5160 5252 5066 5185 4985 5198]
